# Lesson 3a: Training Dynamics — Theory

Lessons 1a/1b derived a single neuron trained by gradient descent, and 2a/2b
stacked neurons into an MLP and derived backpropagation to train it. Both
notebooks trained on the first attempt with an unremarkable initialisation
and plain mini-batch gradient descent — because those networks were small
and shallow. That was not luck so much as scale: the same equations, applied
to a network ten or twenty layers deep, routinely fail outright. The forward
pass produces `NaN`, or the loss simply never moves, even though the
backprop equations are exactly the ones already gradient-checked in 2a.

This notebook is about *why* depth breaks training, and the four ideas that
fix it:

- **Weight initialisation** — the scale of the random weights at the start
  of training determines whether activations shrink to zero or blow up to
  infinity as they pass through many layers, before a single gradient step
  is taken.
- **Vanishing and exploding gradients** — the same shrink/blow-up problem
  applies to the backward pass, and compounds with depth and with the
  choice of activation function.
- **Normalisation layers** — BatchNorm and LayerNorm re-center and re-scale
  activations *during* training, making a network far less sensitive to the
  initialisation and depth problems above.
- **Adaptive optimisers and learning rate schedules** — plain SGD uses one
  global step size for every parameter and every iteration; momentum,
  RMSProp, Adam and learning-rate schedules each relax that in a different,
  motivated way.

By the end of this notebook you will have:
- derived **Xavier/Glorot** and **He/Kaiming** initialisation from a
  variance-preservation argument, and confirmed empirically that they keep
  activation variance stable across depth while naive initialisation does
  not,
- built a **from-scratch deep network** and measured **per-layer gradient
  norms** to watch vanishing and exploding gradients happen,
- derived **BatchNorm** and **LayerNorm** from first principles, implemented
  both **from scratch in NumPy**, and shown what each buys back,
- derived **momentum, RMSProp and Adam** as three successive, motivated
  modifications to the plain SGD update, implemented all four from scratch,
  and compared them on an ill-conditioned toy loss surface,
- derived **step decay, cosine annealing and warmup** learning-rate
  schedules, and
- put all of it together training a small MLP on a **CIFAR-10** subset.

## Introduction

2a/2b's MLP trained on MNIST with a fixed learning rate and small
random weights because a 3-layer network on simple grayscale digits is
forgiving of both choices. Push the same recipe to real images and greater
depth and the forgiveness disappears: this notebook works throughout with
**CIFAR-10** — 32x32 colour photographs across 10 classes, a meaningfully
harder problem than MNIST digits — and with deep stacks purpose-built to
expose failure modes that a 3-layer network never shows.

The rest of this notebook follows the causal chain of the problem, in
order. First: what happens to the *forward* pass, activation by activation,
as a signal repeatedly passes through `randn() * scale`, before any
training happens at all ("Weight Initialisation"). Second: the same
question for the *backward* pass — a per-layer gradient norm plot is the
diagnostic tool used throughout deep learning to answer "is this network
even receiving a training signal at every layer?" ("Vanishing and Exploding
Gradients"). Third, a fix that works *during* training rather than only at
initialisation: normalising activations layer by layer ("Normalisation
Layers"). Fourth and finally, once activations and gradients are
well-behaved, how the *optimiser* turns a gradient into a parameter update,
and why "subtract the gradient times a constant" is the crudest option
available ("Adaptive Optimisers" and "Learning Rate Schedules").

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, data subsampling) is reproducible. Seed both numpy and torch (torch
# is not used for training in this notebook — only NumPy — but CIFAR-10
# decoding uses PIL/pandas which are unaffected by either seed).
import io
import pathlib
import urllib.request

import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)

In [ ]:
# CIFAR-10 via the Hugging Face Datasets Hub parquet mirror, not
# torchvision's `download=True` (which points at www.cs.toronto.edu — a host
# that is frequently rate-limited to a crawl and, unlike torchvision's own
# MNIST mirrors, has no fallback URL). The parquet files decode to the exact
# same 32x32x3 images; only the transport differs, and it works identically
# in Colab and locally.
CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float64) / 255.0
        for i in idx
    ])  # (n, 32, 32, 3), pixel values in [0, 1]
    labels = df.iloc[idx]["label"].to_numpy().astype(int)
    return images, labels


CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

N_TRAIN, N_TEST = 1500, 300
images_train, labels_train = load_cifar10_subset("train", N_TRAIN, seed=SEED)
images_test, labels_test = load_cifar10_subset("test", N_TEST, seed=SEED + 1)

# Flatten to column-per-example, matching 2a's (features, examples) convention.
X_train = images_train.reshape(N_TRAIN, -1).T   # (3072, 1500)
X_test = images_test.reshape(N_TEST, -1).T       # (3072, 300)


def one_hot(labels, n_classes=10):
    Y = np.zeros((n_classes, len(labels)))
    Y[labels, np.arange(len(labels))] = 1.0
    return Y


Y_train = one_hot(labels_train)
Y_test = one_hot(labels_test)

print("X_train:", X_train.shape, " Y_train:", Y_train.shape)
print("X_test: ", X_test.shape, " Y_test: ", Y_test.shape)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(images_train[i])
    ax.set_title(CLASS_NAMES[labels_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10 training samples")
plt.show()

## Weight Initialisation

Consider one linear layer of a network, $z = Wx$, with $x \in
\mathbb{R}^{n_{in}}$, $W \in \mathbb{R}^{n_{out}\times n_{in}}$ initialised
i.i.d. with mean 0 and variance $\sigma_W^2$, and treat the entries of $x$ as
i.i.d. with variance $\sigma_x^2$ and independent of $W$. Each output is a
sum of $n_{in}$ independent zero-mean terms, so its variance adds:

$$\operatorname{Var}(z_j) = \operatorname{Var}\!\left(\sum_{k=1}^{n_{in}} W_{jk}x_k\right)
= \sum_{k=1}^{n_{in}} \operatorname{Var}(W_{jk})\operatorname{Var}(x_k)
= n_{in}\,\sigma_W^2\,\sigma_x^2.$$

Stack $L$ such layers with an activation that is roughly linear near 0 (tanh
is the running example). Every layer multiplies the running variance by the
same factor $n_{in}\sigma_W^2$. Unless that factor is exactly 1, the
activation variance either shrinks geometrically to 0 or grows
geometrically to infinity as depth increases — and it does so *before any
training happens*, purely from the initial random draw. Demanding
$\operatorname{Var}(z) = \operatorname{Var}(x)$ (variance preservation) gives

$$\sigma_W^2 = \frac{1}{n_{in}} \qquad \text{("LeCun" initialisation).}$$

The same argument run **backward** through the same layer — how gradient
variance changes as it propagates from output to input — gives the
analogous condition $\sigma_W^2 = 1/n_{out}$. The two conditions agree only
when $n_{in}=n_{out}$, so **Xavier/Glorot initialisation** compromises by
averaging them:

$$\sigma_W^2 = \frac{2}{n_{in}+n_{out}}
\qquad\Longleftrightarrow\qquad
W_{jk}\sim\mathcal U\!\left(-\sqrt{\tfrac{6}{n_{in}+n_{out}}},\ \sqrt{\tfrac{6}{n_{in}+n_{out}}}\right)$$

for the uniform-distribution form Glorot & Bengio (2010) originally proposed
(a uniform distribution on $[-a,a]$ has variance $a^2/3$, which is where the
$6$ comes from).

**ReLU breaks the "linear near 0" assumption Xavier relies on.** ReLU zeroes
out exactly the negative half of a zero-mean symmetric input, so for
$z\sim\mathcal N(0,\operatorname{Var}(z))$,
$\mathbb E[\text{ReLU}(z)^2] = \tfrac12\operatorname{Var}(z)$: ReLU halves
the variance passed forward on top of the linear-layer scaling above. Redo
the variance-preservation argument through a ReLU layer:

$$\operatorname{Var}(z^{[l+1]}) = n_{in}\,\sigma_W^2 \cdot \tfrac12\operatorname{Var}(z^{[l]})
\ \overset{\text{set}}{=}\ \operatorname{Var}(z^{[l]})
\quad\Longrightarrow\quad
\sigma_W^2 = \frac{2}{n_{in}} \qquad \text{(He/Kaiming initialisation).}$$

He initialisation is exactly Xavier's fan-in variant with the extra factor
of 2 the ReLU nonlinearity demands.

In [ ]:
def init_layer(n_in, n_out, scheme, rng):
    if scheme == "tiny":
        std = 0.01
    elif scheme == "large":
        std = 1.0
    elif scheme == "xavier":
        std = np.sqrt(2.0 / (n_in + n_out))
    elif scheme == "he":
        std = np.sqrt(2.0 / n_in)
    else:
        raise ValueError(scheme)
    return rng.normal(0.0, std, size=(n_out, n_in))


def forward_variance_profile(width, depth, scheme, activation, seed, batch=256):
    """Push random input through `depth` linear+activation layers of a fixed
    `width`, initialised under `scheme`, and record the activation standard
    deviation after every layer (layer 0 = the input itself)."""
    rng = np.random.default_rng(seed)
    a = rng.normal(0.0, 1.0, size=(width, batch))
    stds = [a.std()]
    for _ in range(depth):
        W = init_layer(width, width, scheme, rng)
        z = W @ a
        a = np.tanh(z) if activation == "tanh" else np.maximum(0.0, z)
        stds.append(a.std())
    return np.array(stds)


WIDTH, DEPTH = 128, 40
configs = [
    ("tanh", "tiny", "tanh, std=0.01 (too small)"),
    ("tanh", "large", "tanh, std=1.0 (too large)"),
    ("tanh", "xavier", "tanh, Xavier"),
    ("relu", "tiny", "relu, std=0.01 (too small)"),
    ("relu", "he", "relu, He"),
]

fig, ax = plt.subplots(figsize=(7, 5))
for activation, scheme, label in configs:
    stds = forward_variance_profile(WIDTH, DEPTH, scheme, activation, seed=SEED)
    ax.plot(stds, label=label)
ax.set_yscale("log")
ax.set_xlabel("layer")
ax.set_ylabel("activation std (log scale)")
ax.set_title(f"Forward activation variance vs. depth ({WIDTH}-wide layers)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

Both "too small" curves collapse toward zero within a handful of
layers — by layer 40 the activations carry no usable signal at all. The
"too large" tanh curve does the opposite: it saturates immediately (tanh
caps every output at $\pm 1$, so its std cannot literally explode, but the
*pre-activation* variance does, driving every unit into the flat part of
tanh where its gradient is near zero — the same failure as vanishing,
reached from the other direction). Xavier (tanh) and He (ReLU) both hold the
activation std close to its starting value across all 40 layers, exactly as
the variance-preservation derivation predicts.

## Vanishing and Exploding Gradients

Initialisation only describes the forward pass before training starts.
The backward pass has its own recursion — 2a derived, for a network with
activation $g$,

$$\delta^{[l]} = \left(W^{[l+1]}\right)^{\!\top}\delta^{[l+1]} \odot g'(z^{[l]}),
\qquad \nabla_{W^{[l]}} J = \delta^{[l]}\left(a^{[l-1]}\right)^{\!\top},$$

and this is a product of $L-l$ Jacobian-like terms for a weight at layer
$l$ in an $L$-layer network. If each term in that product has typical
magnitude below 1, the product **vanishes** geometrically with depth; if
above 1, it **explodes**. Two things control the typical magnitude of each
term: the weight scale (exactly the initialisation question above) and the
activation's derivative — sigmoid and tanh have $g'(z)\le 1$ everywhere and
$g'(z)\to 0$ as $|z|$ grows (saturation), so a deep sigmoid/tanh network
multiplies by many sub-1 factors almost everywhere. The diagnostic is to
build one deep network and read off $\lVert\nabla_{W^{[l]}}J\rVert$ layer by
layer.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


ACTS = {
    "sigmoid": (sigmoid, lambda a: a * (1.0 - a)),          # derivative in terms of a=g(z)
    "relu": (lambda z: np.maximum(0.0, z), lambda a: (a > 0.0).astype(np.float64)),
}


def make_deep_net(n_in, width, depth, n_out, scheme, seed):
    rng = np.random.default_rng(seed)
    sizes = [n_in] + [width] * (depth - 1) + [n_out]
    return [init_layer(n_i, n_o, scheme, rng) for n_i, n_o in zip(sizes[:-1], sizes[1:])]


def forward_backward_grad_norms(Ws, X, Y, activation):
    """Forward pass through `Ws`, then backprop a mean-squared-error loss
    against `Y` on the final layer, returning the per-layer weight-gradient
    norm. Generic depth, generic activation — the same recursion as 2a,
    applied L times instead of 3."""
    act, act_prime = ACTS[activation]
    activations = [X]
    a = X
    for W in Ws:
        z = W @ a
        a = act(z)
        activations.append(a)

    L = len(Ws)
    batch = X.shape[1]
    delta = (activations[-1] - Y) * act_prime(activations[-1]) / batch
    grad_norms = [0.0] * L
    for l in range(L - 1, -1, -1):
        grad_W = delta @ activations[l].T
        grad_norms[l] = np.linalg.norm(grad_W)
        if l > 0:
            delta = (Ws[l].T @ delta) * act_prime(activations[l])
    return grad_norms


N_IN, WIDTH, DEPTH_G, N_OUT, BATCH = 64, 64, 30, 10, 128
rng = np.random.default_rng(SEED)
X_toy = rng.normal(size=(N_IN, BATCH))
Y_toy = rng.normal(size=(N_OUT, BATCH)) * 0.1

grad_configs = [
    ("sigmoid", "large", "sigmoid, std=1.0"),
    ("sigmoid", "xavier", "sigmoid, Xavier"),
    ("relu", "tiny", "relu, std=0.01"),
    ("relu", "he", "relu, He"),
]

fig, ax = plt.subplots(figsize=(7, 5))
for activation, scheme, label in grad_configs:
    Ws = make_deep_net(N_IN, WIDTH, DEPTH_G, N_OUT, scheme, seed=SEED)
    norms = forward_backward_grad_norms(Ws, X_toy, Y_toy, activation)
    ax.plot(norms, label=label)
ax.set_yscale("log")
ax.set_xlabel("layer (0 = closest to the input)")
ax.set_ylabel("$\\|\\nabla_{W^{[l]}} J\\|$ (log scale)")
ax.set_title(f"Per-layer gradient norm, {DEPTH_G}-layer network")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

print("gradient norm ratio (last layer / first layer):")
for activation, scheme, label in grad_configs:
    Ws = make_deep_net(N_IN, WIDTH, DEPTH_G, N_OUT, scheme, seed=SEED)
    norms = forward_backward_grad_norms(Ws, X_toy, Y_toy, activation)
    print(f"  {label:22s} {norms[-1] / max(norms[0], 1e-300):.3e}")

Sigmoid at std=1.0 shows the textbook vanishing pattern: the gradient
norm at the layer nearest the input is many orders of magnitude smaller
than at the output layer, because every one of the 30 backward steps
multiplies by a sigmoid derivative that is well below 1 almost everywhere.
Xavier init narrows that gap substantially — sigmoid still saturates
somewhat, so it does not reach a flat line, but it is dramatically better
than the mis-scaled version. ReLU with He initialisation keeps gradient
norms within a much narrower band across all 30 layers: ReLU's derivative is
exactly 0 or exactly 1, so it does not itself shrink a well-scaled gradient,
and He initialisation is precisely the scale that keeps the weight
contribution neutral too.

## Normalisation Layers

Initialisation only fixes the *starting point*; as training proceeds
the weights move and the same shrink/blow-up dynamics can re-emerge layer by
layer. **Batch Normalisation** (Ioffe & Szegedy, 2015) fixes this directly,
during training, by normalising each layer's pre-activation using
statistics computed **across the batch dimension, independently per
feature**. For a batch of $B$ examples and pre-activation $z_{j,i}$
(feature $j$, example $i$):

$$\mu_j = \frac1B\sum_{i=1}^B z_{j,i},\qquad
\sigma_j^2 = \frac1B\sum_{i=1}^B (z_{j,i}-\mu_j)^2,\qquad
\hat z_{j,i} = \frac{z_{j,i}-\mu_j}{\sqrt{\sigma_j^2+\epsilon}},$$

then a learnable per-feature scale and shift restore representational
capacity (without them, every layer would be forced to have exactly
zero mean and unit variance, which is not always what the network needs):
$y_{j,i} = \gamma_j\hat z_{j,i} + \beta_j$. Because $\mu_j,\sigma_j^2$ are
*batch* statistics, BatchNorm needs a reasonably large, representative batch
to estimate them well, and it must behave differently at inference time (a
batch of size 1, or a batch that is not i.i.d. with training data, would
give meaningless statistics) — so BatchNorm tracks a running exponential
average of $\mu_j,\sigma_j^2$ during training and uses that fixed running
average at evaluation time, rather than the current batch's statistics.

**Layer Normalisation** (Ba, Kiros & Hinton, 2016) normalises the *other*
axis: across the feature dimension, independently **per example**, with no
dependence on any other example in the batch:

$$\mu_i = \frac1D\sum_{j=1}^D z_{j,i},\qquad
\sigma_i^2 = \frac1D\sum_{j=1}^D (z_{j,i}-\mu_i)^2,\qquad
\hat z_{j,i} = \frac{z_{j,i}-\mu_i}{\sqrt{\sigma_i^2+\epsilon}},$$

again with a learnable $\gamma,\beta \in \mathbb R^D$ shared across the
batch. Because every quantity in a LayerNorm computation comes from a
single example, it needs no running statistics and no train/eval
distinction — the same formula applies always, even at batch size 1. That
is exactly why LayerNorm, not BatchNorm, is the default in sequence models
and Transformers: batch size 1 (autoregressive generation), variable
sequence length, and non-i.i.d. correlated tokens within a sequence all
make *batch* statistics unreliable or ill-defined, while *per-token,
per-example* statistics are always available. BatchNorm remains the default
in convolutional vision networks, where batches of independent images with
a fixed spatial shape are the norm and its per-channel batch statistics are
well estimated.

In [ ]:
def batchnorm_forward(Z, gamma, beta, eps=1e-5):
    """Z: (features, batch). Normalises across the batch axis, per feature."""
    mu = Z.mean(axis=1, keepdims=True)
    var = Z.var(axis=1, keepdims=True)
    Z_hat = (Z - mu) / np.sqrt(var + eps)
    return gamma * Z_hat + beta, mu, var


def layernorm_forward(Z, gamma, beta, eps=1e-5):
    """Z: (features, batch). Normalises across the feature axis, per example."""
    mu = Z.mean(axis=0, keepdims=True)
    var = Z.var(axis=0, keepdims=True)
    Z_hat = (Z - mu) / np.sqrt(var + eps)
    return gamma * Z_hat + beta, mu, var


def forward_variance_profile_normalised(width, depth, scheme, norm, seed, batch=256):
    """Same experiment as 'Weight Initialisation', but with a normalisation
    layer inserted before the activation at every layer."""
    rng = np.random.default_rng(seed)
    gamma, beta = np.ones((width, 1)), np.zeros((width, 1))
    a = rng.normal(0.0, 1.0, size=(width, batch))
    stds = [a.std()]
    for _ in range(depth):
        W = init_layer(width, width, scheme, rng)
        z = W @ a
        if norm == "batchnorm":
            z, _, _ = batchnorm_forward(z, gamma, beta)
        elif norm == "layernorm":
            z, _, _ = layernorm_forward(z, gamma, beta)
        a = np.tanh(z)
        stds.append(a.std())
    return np.array(stds)


fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(forward_variance_profile(WIDTH, DEPTH, "large", "tanh", seed=SEED),
         label="tanh, std=1.0, no normalisation")
ax.plot(forward_variance_profile_normalised(WIDTH, DEPTH, "large", "batchnorm", seed=SEED),
         label="tanh, std=1.0, + BatchNorm")
ax.plot(forward_variance_profile_normalised(WIDTH, DEPTH, "large", "layernorm", seed=SEED),
         label="tanh, std=1.0, + LayerNorm")
ax.set_yscale("log")
ax.set_xlabel("layer")
ax.set_ylabel("activation std (log scale)")
ax.set_title("Normalisation rescues a badly-scaled initialisation")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

Without normalisation, std=1.0 weights into tanh saturate the network
within a few layers (the same "too large" curve from the initialisation
section). Inserting BatchNorm or LayerNorm before every activation holds
the pre-activation distribution at zero mean, unit variance by
construction, *regardless of the weight scale that produced it* — the
network is no longer at the mercy of getting initialisation exactly right.
This is the practical reason normalisation layers are standard in deep
networks: they make training robust to a design choice (initial weight
scale) that would otherwise have to be tuned precisely.

## Adaptive Optimisers

Every training loop so far has used plain **stochastic gradient
descent (SGD)**: $\theta \leftarrow \theta - \eta\,\nabla_\theta J$. SGD
uses only the current gradient and one global step size $\eta$ for every
parameter. Two problems follow directly. First, loss surfaces are rarely
isotropic — a ravine that is steep in one direction and shallow in another
makes SGD oscillate across the steep direction while crawling along the
shallow one. Second, a single $\eta$ cannot be right for every parameter:
some receive large, frequent gradients and would benefit from a smaller
effective step, others receive small or sparse gradients and would benefit
from a larger one. Momentum, RMSProp and Adam are three successive,
independently motivated fixes.

**Momentum** addresses the oscillation problem by accumulating a running
average of past gradients, so that consistent directions build up speed and
oscillating directions cancel out:

$$v \leftarrow \beta v + (1-\beta)\nabla_\theta J, \qquad \theta \leftarrow \theta - \eta v.$$

**RMSProp** addresses the per-parameter step-size problem instead, by
dividing each parameter's step by a running estimate of that parameter's
own gradient magnitude:

$$s \leftarrow \beta s + (1-\beta)\left(\nabla_\theta J\right)^2, \qquad
\theta \leftarrow \theta - \frac{\eta}{\sqrt s + \epsilon}\,\nabla_\theta J.$$

A parameter with a history of large gradients gets a large $s$ and hence a
small effective step; a parameter with small or sparse gradients gets a
large effective step — no manual per-parameter tuning required.

**Adam** (Kingma & Ba, 2015) combines both fixes — momentum's first moment
$m$ and RMSProp's second moment $v$ — and adds a bias correction, because
$m$ and $v$ start at exactly 0 and are therefore biased toward 0 for the
first few steps (most severely when $\beta_1,\beta_2$ are close to 1):

$$m \leftarrow \beta_1 m + (1-\beta_1)\nabla_\theta J, \qquad
v \leftarrow \beta_2 v + (1-\beta_2)\left(\nabla_\theta J\right)^2,$$

$$\hat m = \frac{m}{1-\beta_1^t}, \qquad \hat v = \frac{v}{1-\beta_2^t},
\qquad \theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat v}+\epsilon}\,\hat m.$$

Each optimiser below is a strict superset of the update rule before it:
Momentum is SGD plus a running average of the gradient; RMSProp is SGD plus
a running average of the squared gradient; Adam is both running averages
together, bias-corrected.

In [ ]:
class SGD:
    def __init__(self, lr=0.1):
        self.lr = lr

    def step(self, params, grads, state):
        for k in params:
            params[k] -= self.lr * grads[k]
        return state


class Momentum:
    def __init__(self, lr=0.1, beta=0.9):
        self.lr, self.beta = lr, beta

    def step(self, params, grads, state):
        state = state or {k: np.zeros_like(v) for k, v in params.items()}
        for k in params:
            state[k] = self.beta * state[k] + (1 - self.beta) * grads[k]
            params[k] -= self.lr * state[k]
        return state


class RMSProp:
    def __init__(self, lr=0.01, beta=0.9, eps=1e-8):
        self.lr, self.beta, self.eps = lr, beta, eps

    def step(self, params, grads, state):
        state = state or {k: np.zeros_like(v) for k, v in params.items()}
        for k in params:
            state[k] = self.beta * state[k] + (1 - self.beta) * grads[k] ** 2
            params[k] -= self.lr * grads[k] / (np.sqrt(state[k]) + self.eps)
        return state


class Adam:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps

    def step(self, params, grads, state):
        if state is None:
            state = {"t": 0, "m": {k: np.zeros_like(v) for k, v in params.items()},
                      "v": {k: np.zeros_like(v) for k, v in params.items()}}
        state["t"] += 1
        t = state["t"]
        for k in params:
            state["m"][k] = self.beta1 * state["m"][k] + (1 - self.beta1) * grads[k]
            state["v"][k] = self.beta2 * state["v"][k] + (1 - self.beta2) * grads[k] ** 2
            m_hat = state["m"][k] / (1 - self.beta1 ** t)
            v_hat = state["v"][k] / (1 - self.beta2 ** t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return state

A classic diagnostic for exactly the oscillation problem Momentum
targets is an **ill-conditioned quadratic bowl**, $f(x,y)=\tfrac12(a x^2 +
b y^2)$ with $a \gg b$: steep in $x$, shallow in $y$. Its gradient is
$\nabla f = (ax,\ by)$.

In [ ]:
A_COEF, B_COEF = 60.0, 1.0


def bowl_loss(theta):
    return 0.5 * (A_COEF * theta["x"] ** 2 + B_COEF * theta["y"] ** 2)


def bowl_grad(theta):
    return {"x": A_COEF * theta["x"], "y": B_COEF * theta["y"]}


def run_optimiser(opt, n_steps=60, start=(1.0, 1.0)):
    theta = {"x": np.array(start[0]), "y": np.array(start[1])}
    state = None
    path = [(float(theta["x"]), float(theta["y"]))]
    losses = [float(bowl_loss(theta))]
    for _ in range(n_steps):
        grads = bowl_grad(theta)
        state = opt.step(theta, grads, state)
        path.append((float(theta["x"]), float(theta["y"])))
        losses.append(float(bowl_loss(theta)))
    return np.array(path), np.array(losses)


optimisers = {
    "SGD": SGD(lr=0.016),
    "Momentum": Momentum(lr=0.016, beta=0.9),
    "RMSProp": RMSProp(lr=0.3),
    "Adam": Adam(lr=0.3),
}

fig, (ax_path, ax_loss) = plt.subplots(1, 2, figsize=(12, 5))

xs = np.linspace(-1.2, 1.2, 200)
ys = np.linspace(-1.2, 1.2, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.5 * (A_COEF * XX ** 2 + B_COEF * YY ** 2)
ax_path.contour(XX, YY, ZZ, levels=20, cmap="Greys", alpha=0.5)

for name, opt in optimisers.items():
    path, losses = run_optimiser(opt)
    ax_path.plot(path[:, 0], path[:, 1], marker=".", markersize=3, label=name)
    ax_loss.plot(losses, label=name)

ax_path.set_xlabel("x (steep direction)")
ax_path.set_ylabel("y (shallow direction)")
ax_path.set_title(f"Optimiser trajectories on an ill-conditioned bowl (a/b={A_COEF/B_COEF:.0f})")
ax_path.legend(fontsize=8)

ax_loss.set_yscale("log")
ax_loss.set_xlabel("step")
ax_loss.set_ylabel("loss (log scale)")
ax_loss.set_title("Convergence")
ax_loss.legend(fontsize=8)
ax_loss.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Plain SGD zig-zags visibly across the steep $x$ direction while making
slow progress along the shallow $y$ direction — exactly the oscillation the
derivation predicted. Momentum damps the zig-zag by averaging out the
sign-flipping $x$-gradient while the consistent $y$-gradient accumulates,
reaching the minimum in noticeably fewer steps. RMSProp and Adam divide
each direction's step by its own gradient history, which shrinks the large,
oscillating $x$-steps and grows the small, steady $y$-steps automatically —
both converge fastest here, with Adam adding momentum's smoothing on top of
RMSProp's per-parameter scaling.

Now the same four optimisers on the actual task: a small MLP classifying
the CIFAR-10 subset loaded in "Setup". This reuses the plain-NumPy MLP
forward/backward structure from 2a, with He initialisation and one
BatchNorm layer (both justified above) so that any remaining difference in
the loss curves is attributable to the optimiser, not to a bad
initialisation confounding the comparison.

In [ ]:
def relu(z):
    return np.maximum(0.0, z)


def relu_prime(a):
    return (a > 0.0).astype(np.float64)


def softmax(z):
    z = z - z.max(axis=0, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=0, keepdims=True)


class CifarMLP:
    """[3072, 256, 64, 10], He-initialised, BatchNorm after the first linear
    layer. Forward/backward follow 2a's derivation; BatchNorm's forward
    pass and its backward pass (through the normalisation, not just the
    affine gamma/beta) are added at the one layer that uses it."""

    def __init__(self, seed):
        rng = np.random.default_rng(seed)
        sizes = [3072, 256, 64, 10]
        self.W = [init_layer(n_i, n_o, "he", rng) for n_i, n_o in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros((n_o, 1)) for n_o in sizes[1:]]
        self.gamma = np.ones((sizes[1], 1))
        self.beta = np.zeros((sizes[1], 1))
        self.running_mu = np.zeros((sizes[1], 1))
        self.running_var = np.ones((sizes[1], 1))

    def forward(self, X, train=True, momentum=0.9, eps=1e-5):
        cache = {"a0": X}
        z1 = self.W[0] @ X + self.b[0]
        if train:
            mu = z1.mean(axis=1, keepdims=True)
            var = z1.var(axis=1, keepdims=True)
            self.running_mu = momentum * self.running_mu + (1 - momentum) * mu
            self.running_var = momentum * self.running_var + (1 - momentum) * var
        else:
            mu, var = self.running_mu, self.running_var
        z1_hat = (z1 - mu) / np.sqrt(var + eps)
        z1_bn = self.gamma * z1_hat + self.beta
        a1 = relu(z1_bn)
        cache.update(z1=z1, mu=mu, var=var, z1_hat=z1_hat, a1=a1)

        z2 = self.W[1] @ a1 + self.b[1]
        a2 = relu(z2)
        cache.update(z2=z2, a2=a2)

        z3 = self.W[2] @ a2 + self.b[2]
        a3 = softmax(z3)
        cache.update(z3=z3, a3=a3)
        return a3, cache

    def backward(self, Y, cache, eps=1e-5):
        B = Y.shape[1]
        grads = {}
        delta3 = (cache["a3"] - Y) / B
        grads["W3"] = delta3 @ cache["a2"].T
        grads["b3"] = delta3.sum(axis=1, keepdims=True)

        delta2 = (self.W[2].T @ delta3) * relu_prime(cache["a2"])
        grads["W2"] = delta2 @ cache["a1"].T
        grads["b2"] = delta2.sum(axis=1, keepdims=True)

        delta_a1 = self.W[1].T @ delta2
        delta_bn = delta_a1 * relu_prime(cache["a1"])          # through ReLU
        grads["gamma"] = (delta_bn * cache["z1_hat"]).sum(axis=1, keepdims=True)
        grads["beta"] = delta_bn.sum(axis=1, keepdims=True)
        dz1_hat = delta_bn * self.gamma
        std_inv = 1.0 / np.sqrt(cache["var"] + eps)
        z1_centered = cache["z1"] - cache["mu"]
        dvar = (dz1_hat * z1_centered * -0.5 * std_inv ** 3).sum(axis=1, keepdims=True)
        dmu = (dz1_hat * -std_inv).sum(axis=1, keepdims=True) + \
              dvar * (-2.0 * z1_centered).mean(axis=1, keepdims=True)
        delta1 = dz1_hat * std_inv + dvar * 2.0 * z1_centered / B + dmu / B  # through BatchNorm

        grads["W1"] = delta1 @ cache["a0"].T
        grads["b1"] = delta1.sum(axis=1, keepdims=True)
        return grads

    def params(self):
        return {"W1": self.W[0], "b1": self.b[0], "gamma": self.gamma, "beta": self.beta,
                "W2": self.W[1], "b2": self.b[1], "W3": self.W[2], "b3": self.b[2]}

    def set_params(self, p):
        self.W[0], self.b[0], self.gamma, self.beta = p["W1"], p["b1"], p["gamma"], p["beta"]
        self.W[1], self.b[1] = p["W2"], p["b2"]
        self.W[2], self.b[2] = p["W3"], p["b3"]


def cross_entropy(probs, Y):
    return -np.mean(np.sum(Y * np.log(np.clip(probs, 1e-12, 1.0)), axis=0))


def accuracy(model, X, labels):
    probs, _ = model.forward(X, train=False)
    return (probs.argmax(axis=0) == labels).mean()


def train_cifar_mlp(opt, epochs=12, batch_size=64, seed=SEED):
    model = CifarMLP(seed=seed)
    g = np.random.default_rng(seed)
    n = X_train.shape[1]
    state = None
    losses = []
    for _ in range(epochs):
        order = g.permutation(n)
        epoch_losses = []
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            probs, cache = model.forward(X_train[:, batch], train=True)
            epoch_losses.append(cross_entropy(probs, Y_train[:, batch]))
            grads = model.backward(Y_train[:, batch], cache)
            params = model.params()
            state = opt.step(params, grads, state)
            model.set_params(params)
        losses.append(np.mean(epoch_losses))
    return model, losses


optimiser_factories = {
    "SGD": lambda: SGD(lr=0.5),
    "Momentum": lambda: Momentum(lr=0.2, beta=0.9),
    "RMSProp": lambda: RMSProp(lr=0.005),
    "Adam": lambda: Adam(lr=0.003),
}

fig, ax = plt.subplots(figsize=(7, 5))
final_test_acc = {}
for name, factory in optimiser_factories.items():
    model, losses = train_cifar_mlp(factory(), epochs=20)
    ax.plot(losses, label=name)
    final_test_acc[name] = accuracy(model, X_test, labels_test)
ax.set_xlabel("epoch")
ax.set_ylabel("training cross-entropy loss")
ax.set_title("CIFAR-10 subset: optimiser comparison (same init, same BatchNorm)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

for name, acc in final_test_acc.items():
    print(f"  {name:10s} test accuracy: {acc:.3f}")
assert final_test_acc["Adam"] > 0.2, "Adam should clear well above the 10% chance level"

Adam and RMSProp drive the training loss down fastest and reach the
highest test accuracy in the fixed epoch budget, plain SGD is visibly
slower to move, and Momentum sits between the two — the same ordering the
toy bowl predicted, now on a real (if small) classification problem, with a
BatchNorm layer already doing the initialisation-robustness work from the
previous section.

## Learning Rate Schedules

A single fixed $\eta$ is itself another crude simplification: early in
training, when Adam's moment estimates are least reliable (their bias
correction divides by $1-\beta^t$, which is largest exactly when $t$ is
small), a large step size that is fine later can be actively harmful. Three
schedules address this, each a function $\eta_t$ of the step or epoch $t$:

**Step decay** — hold $\eta_0$ for a fixed number of epochs, then drop it by
a constant factor, repeatedly:

$$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / s\rfloor}$$

for decay factor $\gamma<1$ and step size $s$. Simple, but the drops are
discontinuous and $s,\gamma$ must be chosen per problem.

**Cosine annealing** (Loshchilov & Hutter, 2017) replaces the discontinuous
drop with a smooth decay from $\eta_0$ to $\eta_{min}$ following one half
cosine cycle over the total training horizon $T$:

$$\eta_t = \eta_{min} + \tfrac12(\eta_0-\eta_{min})\left(1+\cos\!\left(\pi t/T\right)\right).$$

**Warmup** addresses the *opposite* end of training: rather than starting
at the full $\eta_0$ immediately, ramp linearly from 0 up to $\eta_0$ over
the first $W$ steps, then hand off to whatever schedule (constant, step,
cosine) follows:

$$\eta_t = \eta_0 \cdot \frac{t}{W} \quad \text{for } t < W,
\qquad \text{then continue with the chosen post-warmup schedule.}$$

Warmup is standard practice with Adam specifically for the bias-correction
reason above: a large step multiplied by an unreliable early moment
estimate is exactly the recipe for an unstable first few iterations.

In [ ]:
def step_decay(t, eta0=0.01, gamma=0.5, step_size=10):
    return eta0 * gamma ** (t // step_size)


def cosine_annealing(t, eta0=0.01, eta_min=0.0005, total_steps=60):
    return eta_min + 0.5 * (eta0 - eta_min) * (1 + np.cos(np.pi * min(t, total_steps) / total_steps))


def warmup_cosine(t, eta0=0.01, eta_min=0.0005, warmup_steps=5, total_steps=60):
    if t < warmup_steps:
        return eta0 * (t + 1) / warmup_steps
    return cosine_annealing(t - warmup_steps, eta0, eta_min, total_steps - warmup_steps)


T_EPOCHS = 60
ts = np.arange(T_EPOCHS)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ts, [step_decay(t) for t in ts], label="step decay")
ax.plot(ts, [cosine_annealing(t) for t in ts], label="cosine annealing")
ax.plot(ts, [warmup_cosine(t) for t in ts], label="warmup + cosine")
ax.set_xlabel("epoch")
ax.set_ylabel("learning rate")
ax.set_title("Learning rate schedules")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
class AdamScheduled(Adam):
    """Adam whose learning rate is looked up from a schedule function of the
    current step, rather than held fixed — a one-line change to the
    optimiser derived above."""

    def __init__(self, schedule, beta1=0.9, beta2=0.999, eps=1e-8):
        super().__init__(lr=0.0, beta1=beta1, beta2=beta2, eps=eps)
        self.schedule = schedule
        self.epoch = 0

    def step(self, params, grads, state):
        self.lr = self.schedule(self.epoch)
        return super().step(params, grads, state)


def train_cifar_mlp_scheduled(opt, epochs=20, batch_size=64, seed=SEED):
    model = CifarMLP(seed=seed)
    g = np.random.default_rng(seed)
    n = X_train.shape[1]
    state = None
    losses = []
    for epoch in range(epochs):
        if hasattr(opt, "epoch"):
            opt.epoch = epoch
        order = g.permutation(n)
        epoch_losses = []
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            probs, cache = model.forward(X_train[:, batch], train=True)
            epoch_losses.append(cross_entropy(probs, Y_train[:, batch]))
            grads = model.backward(Y_train[:, batch], cache)
            params = model.params()
            state = opt.step(params, grads, state)
            model.set_params(params)
        losses.append(np.mean(epoch_losses))
    return model, losses


EPOCHS_SCHED = 20
fixed_model, fixed_losses = train_cifar_mlp_scheduled(Adam(lr=0.003), epochs=EPOCHS_SCHED)
sched_opt = AdamScheduled(lambda t: warmup_cosine(t, eta0=0.006, eta_min=0.0005,
                                                    warmup_steps=3, total_steps=EPOCHS_SCHED))
sched_model, sched_losses = train_cifar_mlp_scheduled(sched_opt, epochs=EPOCHS_SCHED)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fixed_losses, label="Adam, fixed lr=0.003")
ax.plot(sched_losses, label="Adam, warmup + cosine (peak lr=0.006)")
ax.set_xlabel("epoch")
ax.set_ylabel("training cross-entropy loss")
ax.set_title("CIFAR-10 subset: fixed learning rate vs. warmup + cosine")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

print(f"fixed-lr final test accuracy:      {accuracy(fixed_model, X_test, labels_test):.3f}")
print(f"warmup+cosine final test accuracy: {accuracy(sched_model, X_test, labels_test):.3f}")

The scheduled run starts at a lower learning rate during warmup (safer
while Adam's moment estimates are still unreliable), reaches a higher peak
learning rate than the fixed run ever uses (faster progress once those
estimates have stabilised), and anneals smoothly toward zero as training
ends (fine-tuning around the minimum rather than bouncing past it) —
combining the benefits of a larger step size with the stability of a small
one, at the two ends of training where each is needed.

## Key Takeaways

- **Weight initialisation** must preserve activation variance across
  depth or the forward pass vanishes or explodes before any training
  happens: $\sigma_W^2=2/(n_{in}+n_{out})$ for tanh/sigmoid (**Xavier**),
  $\sigma_W^2=2/n_{in}$ for ReLU (**He**), both derived from the same
  variance-preservation argument that a naive $\sigma_W^2=0.01$ or $1.0$
  violates.
- **Vanishing and exploding gradients** are the backward-pass version of
  the same problem: the gradient at layer $l$ is a product of $L-l$
  Jacobian-like terms, so any systematic sub-1 or super-1 factor (a
  saturating activation, a badly-scaled weight) compounds geometrically
  with depth — visible directly as a per-layer gradient-norm plot.
- **BatchNorm** normalises across the batch, per feature, and needs running
  statistics for train/eval consistency; **LayerNorm** normalises across
  features, per example, needs no running statistics, and works at batch
  size 1 — which is why BatchNorm is standard in vision CNNs and LayerNorm
  in sequence models and Transformers. Both rescue a network from a
  mis-scaled initialisation *during* training, not just at $t=0$.
- **Momentum, RMSProp and Adam** are three independently motivated,
  strictly additive fixes to plain SGD: momentum averages the gradient
  itself to damp oscillation along steep directions, RMSProp divides by a
  running estimate of each parameter's own gradient magnitude to equalise
  effective step sizes, and Adam combines both with a bias correction for
  their first few, otherwise-biased-toward-zero, updates.
- **Learning rate schedules** — step decay, cosine annealing, and warmup —
  address a different single-$\eta$ crudeness: too large a step is unstable
  early (especially with Adam's unreliable early moment estimates) and
  wastes potential progress late, when a small step lets the optimiser
  settle near a minimum instead of bouncing around it.
- Every idea in this notebook — initialisation, normalisation, adaptive
  optimisers, schedules — is orthogonal to the backpropagation equations
  derived in 2a: none of them change what the gradient *is*, only how
  reliably it is computed (initialisation, normalisation) or how it is
  turned into a parameter update (optimisers, schedules). Lesson 3b
  reproduces every one of these effects with PyTorch's built-in
  `nn.init`, `nn.BatchNorm1d`/`nn.LayerNorm`, and `torch.optim` — this
  notebook derived and hand-built exactly what those calls do
  internally.